# MedSpeak YO→EN — Whisper-Medium ASR Fine-Tuning (Colab T4)

Fine-tunes **Whisper Medium** for Yorùbá ASR on a **pooled** corpus (FLEURS yo + OpenSLR SLR86), with on-the-fly audio augmentation. This is the *real* fix for the yo→en speech direction: the deployed Whisper-small was trained on FLEURS alone, which is why end-to-end BLEU lags the clean-text NMT BLEU.

**Before you run anything:** `Runtime → Change runtime type → Hardware accelerator: GPU (T4)`.

Checkpoints are written to Google Drive, so a dropped session never costs more than one epoch — just re-run the training cell and it auto-resumes (`[RESUME] continuing from …`).

FLEURS **test** is held out for WER comparability with the existing baseline (whisper-medium zero-shot 103.65% → whisper-small fine-tuned 63.40%).

## 1. Confirm the GPU
Medium needs ~13–15 GB during training. A T4 (16 GB) is enough with batch 2 + grad-accum 8 + fp16 + gradient checkpointing + `expandable_segments` (all set by the script / training cell).

In [ ]:
!nvidia-smi

## 2. Clone the repo

In [ ]:
!git clone https://github.com/eddy-udoh/fyp-yoruba-english-s2st.git
%cd fyp-yoruba-english-s2st
# The ASR training code currently lives on this branch.
# Once the PR is merged to main, you can delete this checkout line.
!git checkout asr/whisper-medium-pooled
!git log --oneline -1

## 3. Install training dependencies
Only what ASR fine-tuning needs. `torch` ships with Colab. `datasets` pinned `<3.0` (3.x dropped `trust_remote_code` / script datasets, which SLR86 + FLEURS rely on).

In [ ]:
!pip install -q -U transformers "datasets<3.0" accelerate jiwer soundfile librosa
# librosa is needed by `datasets` to resample audio to 16 kHz on decode.

## 4. Mount Google Drive (checkpoint persistence)
The model + checkpoints are written straight to Drive so they survive a disconnect and you keep them after the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUT_DIR = '/content/drive/MyDrive/whisper-medium-yoruba-finetuned'
print('whisper checkpoints ->', OUT_DIR)

## 5. (Optional) Smoke test — prove the pipeline in ~2 min
Tiny model, 32 clips/source, 1 epoch, no augmentation. Confirms data loading → collator → trainer → eval → JSON all work before you commit to the full run. Skip if you're confident.

In [ ]:
!python src/asr/whisper_finetune.py --smoke

## 6. Fine-tune Whisper Medium (the real run)
Pools FLEURS yo + SLR86, augments audio (noise/gain/speed), checkpoints to Drive every ~200 steps. **~2–3 h on a T4** depending on early stopping.

**Memory:** medium sits right at the T4's 16 GB ceiling. This is set to `--batch-size 2 --grad-accum 8` (effective batch 16, same as 4×4 but half the activation memory) because 4×4 OOMs partway through. The script also bakes in `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` to avoid fragmentation OOMs. If you still OOM, go to `--batch-size 1 --grad-accum 16`.

**Resuming:** if the session drops, just re-run this cell — it prints `[RESUME] continuing from …/checkpoint-N` and picks up from the last ~200-step checkpoint on Drive (you lose minutes, not the whole run).

Watch the **eval WER** (every 200 steps) and the **FLEURS-test WER** printed at the end (goal: beat 63.40%).

In [ ]:
!python src/asr/whisper_finetune.py \
    --model-size medium \
    --augment \
    --batch-size 2 \
    --grad-accum 8 \
    --output-dir "$OUT_DIR"

## 7. (Optional) Add more Yorùbá hours / in-domain audio
- Pool another **verified** HF source (e.g. ÌròyìnSpeech once you confirm its repo id) with `--extra-hf repo:config:split:audio_col:text_col` (repeatable).
- Pool local **clinical** audio you've obtained with `--clinical-manifest CSV(audio_path,sentence)`, and hold out a clinical test split with `--clinical-test-manifest`.

Example (edit before running):

In [ ]:
# !python src/asr/whisper_finetune.py --model-size medium --augment \
#     --batch-size 4 --grad-accum 4 --output-dir "$OUT_DIR" \
#     --extra-hf "REPO_ID:CONFIG:train:audio:sentence"

## 8. Save eval results to Drive & verify the model files

In [ ]:
!mkdir -p /content/drive/MyDrive/fyp_eval
!cp evaluation/asr_finetuned_whisper-*.json /content/drive/MyDrive/fyp_eval/ 2>/dev/null; echo 'eval JSON copied'
print('\n--- whisper model files ---')
!ls -lh "$OUT_DIR" | grep -E 'safetensors|config.json|preprocessor|tokenizer|vocab|merges'

## 9. Integrate back into the repo (on your local machine)

The trained weights live on your Drive. To deploy:

1. Download the `whisper-medium-yoruba-finetuned` folder from Drive into `models/` in the repo.
2. In **`src/api/app.py`** and **`src/eval/end_to_end_eval.py`**, bump the two constants:
   - `WHISPER_MODEL_PATH = os.path.join(MODELS_DIR, "whisper-medium-yoruba-finetuned")`
   - `WHISPER_BASE_ID    = "openai/whisper-medium"`
3. Re-run `src/eval/end_to_end_eval.py` — this is where the end-to-end BLEU should climb.

> The weights are git-ignored (too large for GitHub), so keep the Drive copy as your backup.